In [1]:
import pandas as pd
import os
import numpy as np
import torch
import faiss
from sentence_transformers import SentenceTransformer

In [3]:
os.chdir('/home/rahul_goyal/product-debater-video-games/')

In [4]:
df_reviews = pd.read_parquet("data/reviews_clean.parquet", dtype_backend="pyarrow")
df_meta = pd.read_parquet("data/meta_clean.parquet", dtype_backend="pyarrow")

In [5]:
df_reviews.head()

,rating,title,text,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,review_length
0,4,It’s pretty sexual. Not my fav,I’m playing on ps5 and it’s interesting. It’s...,B07DJWBYKP,B07DK1H3H5,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,2020-12-17 06:33:24.795000,0,True,42
3,5,Great alt to pro controller,"These work great, They use batteries which is ...",B07GXJHRVK,B0BCHWZX95,AFTC6ZR5IKNRDG5JCPVNVMU3XV2Q,2019-12-29 16:40:34.017000,0,True,39
5,3,love all the amazing colors but the black is r...,love all the amazing colors but the black is r...,B016Y2BVKA,B073SC6V1D,AHXSBZT52TCPZUBVCBRICTHWUCBA,2018-02-08 21:15:39.574000,0,True,31
25,5,Good Chocolates,Husband bought these for me for our 25th Anniv...,B07W738MG5,B00HBUOVIO,AE7Y5RLYIKHOZB5NKKOEKYG2SPSQ,2022-12-26 19:38:06.117000,0,True,30
30,5,Nintendo Switch,I actually purchased this so that I could have...,B07VGRJDFY,B07VGRJDFY,AFZUK3MTBIBEDQOPAK3OATUOUKLA,2021-01-28 05:17:34.257000,0,True,153


In [6]:
df_semantic = df_reviews[df_reviews['review_length'] >= 25].copy()

In [7]:
# CRITICAL: Reset index so row numbers perfectly match FAISS vector IDs
df_semantic = df_semantic.reset_index(drop=True)
print(f"Target semantic reviews to embed: {len(df_semantic):,}")

Target semantic reviews to embed: 396,080


In [8]:
# 1. Hardware Verification
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Compute Device: {device.upper()}")
if device == "cuda":
    print(f"GPU Detected: {torch.cuda.get_device_name(0)}")

Compute Device: CUDA
GPU Detected: NVIDIA GeForce RTX 4060 Laptop GPU


In [9]:
print("Loading BAAI BGE-M3 model into VRAM...")
model = SentenceTransformer(
    "BAAI/bge-m3", 
    device=device,
    model_kwargs={"torch_dtype": torch.float16} 
)

Loading BAAI BGE-M3 model into VRAM...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [10]:
sum(df_semantic['review_length'] >= 1024)/len(df_semantic)

0.0020728135730155523

In [11]:
print(next(model.parameters()).device)

cuda:0


In [12]:
print("Applying hard string truncation to 1024 words...")
# Preserving the massive data context you wanted
texts = [" ".join(text.split()[:1024]) for text in df_semantic['text'].tolist()]

print("Generating embeddings... (Watch the progress bar)")
embeddings = model.encode(
    texts, 
    batch_size=64, # Kept at 16 to safely process 1024-word blocks on 8GB VRAM
    show_progress_bar=True, 
    normalize_embeddings=True
)

Applying hard string truncation to 1024 words...
Generating embeddings... (Watch the progress bar)


Batches:   0%|          | 0/6189 [00:00<?, ?it/s]

In [13]:
# 5. Build FAISS
print("Building FAISS Vector Database...")
index = faiss.IndexFlatIP(embeddings.shape[1]) 
index.add(embeddings)

Building FAISS Vector Database...


In [14]:
faiss.write_index(index, "data/video_games_bge.index")
df_semantic.to_parquet("data/semantic_metadata.parquet", engine="pyarrow")

print("\n=== PIPELINE COMPLETE ===")
print("Saved: data/video_games_bge.index")
print("Saved: data/semantic_metadata.parquet")


=== PIPELINE COMPLETE ===
Saved: data/video_games_bge.index
Saved: data/semantic_metadata.parquet
